<a href="https://colab.research.google.com/github/nurulashraf/spotify-user-library/blob/main/spotify_user_data_retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import webbrowser
from pathlib import Path
from dotenv import load_dotenv
import os

# Load credentials from .env (project root = spotify-data-retrieval folder)
env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)
client_id = os.getenv("SPOTIFY_CLIENT_ID")
client_secret = os.getenv("SPOTIFY_CLIENT_SECRET")
redirect_uri = os.getenv("SPOTIFY_REDIRECT_URI", "http://127.0.0.1:8080/callback")
scopes = "user-library-read user-top-read user-read-playback-state"

auth_url = (
    f"https://accounts.spotify.com/authorize"
    f"?client_id={client_id}"
    f"&response_type=code"
    f"&redirect_uri={redirect_uri}"
    f"&scope={scopes}"
)
print(auth_url)
webbrowser.open(auth_url)

In [ ]:
import requests
import base64
from pathlib import Path
from dotenv import load_dotenv
import os

# Load credentials from .env
env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)
client_id = os.getenv("SPOTIFY_CLIENT_ID")
client_secret = os.getenv("SPOTIFY_CLIENT_SECRET")
redirect_uri = os.getenv("SPOTIFY_REDIRECT_URI", "http://127.0.0.1:8080/callback")

# Paste the code from the URL after you authorized (the part after ?code= and before &)
auth_code = 'PASTE_CODE_HERE'

# Encode the client ID and client secret
auth_str = f"{client_id}:{client_secret}"
auth_bytes = auth_str.encode('utf-8')
auth_base64 = base64.b64encode(auth_bytes).decode('utf-8')

# Exchange code for token
auth_url = 'https://accounts.spotify.com/api/token'
headers = {
    'Authorization': 'Basic ' + auth_base64,
    'Content-Type': 'application/x-www-form-urlencoded'
}
data = {
    'grant_type': 'authorization_code',
    'code': auth_code,
    'redirect_uri': redirect_uri
}

response = requests.post(auth_url, headers=headers, data=data)
tokens = response.json()
access_token = tokens['access_token']

# Function to get saved albums
def get_saved_albums():
    endpoint = 'https://api.spotify.com/v1/me/albums'
    headers = {
        'Authorization': f'Bearer {access_token}'
    }

    response = requests.get(endpoint, headers=headers)
    return response.json()

# Function to get liked songs
def get_liked_songs():
    endpoint = 'https://api.spotify.com/v1/me/tracks'
    headers = {
        'Authorization': f'Bearer {access_token}'
    }

    response = requests.get(endpoint, headers=headers)
    return response.json()

# Function to get playlists
def get_playlists():
    endpoint = 'https://api.spotify.com/v1/me/playlists'
    headers = {
        'Authorization': f'Bearer {access_token}'
    }

    response = requests.get(endpoint, headers=headers)
    return response.json()

# Fetch and display user data
saved_albums = get_saved_albums()
print("User's Saved Albums:")
for album in saved_albums['items']:
    print(f" - {album['album']['name']} by {', '.join(artist['name'] for artist in album['album']['artists'])}")

liked_songs = get_liked_songs()
print("\nUser's Liked Songs:")
for song in liked_songs['items']:
    print(f" - {song['track']['name']} by {', '.join(artist['name'] for artist in song['track']['artists'])}")

playlists = get_playlists()
print("\nUser's Playlists:")
for playlist in playlists['items']:
    print(f" - {playlist['name']} (ID: {playlist['id']})")
